# Thymio autonomous parking

MICRO-452 — Basics of mobile robotics — project report

## 1. Group members and tasks

| Member | Task |
|---|---|
| Joshua | Vision, filtering |
| Melvyn | Global navigation |
| Antoine | Motion control |
| Laurent | Local navigation |

*To complete: full names, and who did what in the end.*

## 2. Environment and choices

*To write: the parking lot, the obstacles, the algorithms we chose and why.*

In [ ]:
# Run this notebook from the root of the repository.
from src import common

## 3. Vision

*To write (Joshua): theory, choices, measurements, parameters. Run the block alone on images from `data/`.*

## 4. Global navigation

*To write (Melvyn): theory, choices, parameters. Run the block alone on example grids.*

## 5. Motion control

*To write (Antoine): theory, choices, measurements, parameters. Run the block alone.*

## 6. Local navigation


### 6.1. Overview

The local navigation module handles unexpected obstacles that are not considered during global path planning.

It uses the Thymio's five front-facing proximity sensors to detect obstacles and select an avoidance direction. The two ground sensors are also used to detect the black boundary of the parking area.

The module can temporarily override global navigation commands and return control once the obstacle has been cleared.

### 6.2. Architecture choice

Initially, we chose to implement the entire navigation pipeline on the PC rather than using a hybrid architecture with local navigation running onboard the Thymio.

Although onboard execution could improve reaction time, it would introduce additional complexity related to communication, synchronization and debugging.

A fully PC-based implementation therefore provides a simpler environment for developing and validating the complete system. A hybrid architecture remains a possible future improvement if communication latency becomes a limitation.

### 6.3. Obstacle detection and threshold calibration

The proximity sensor values increase as an obstacle gets closer. Obstacle detection is based on a threshold \(T\):

$\exists i \in \{0,\ldots,4\}: prox_i > T$

Two thresholds are used depending on the navigation state:

- **FOLLOW_THRESHOLD:** lower threshold, allowing earlier obstacle detection during normal path following.
- **PARKING_THRESHOLD:** higher threshold, allowing the robot to approach parked vehicles more closely.


In [1]:
def obstacle_detected(prox, threshold):
    sensors = [value > threshold for value in prox[:5]]
    return any(sensors), sensors

To illustrate the behaviour of the function, we can simulate a situation where an obstacle is detected by the central front sensor. The other sensors return values below the detection threshold

In [3]:
# Simulated proximity sensor readings
prox = [1200, 1800, 3500, 2100, 1500, 0, 0]
threshold = 3000 #FOLLOW_THRESHOLD

obstacle, sensors = obstacle_detected(prox, threshold)

print("Individual sensors:", sensors)
print("Obstacle detected:", obstacle)

Individual sensors: [False, False, True, False, False]
Obstacle detected: True


To support this behaviour, the `avoid()` interface was extended with a third parameter:

```python
avoid(prox, ground, threshold)
```

This allows a high level planning done by the motion control module.
Here's the measurments that we did : for the record we computed the best polynomial fit :

$ 
d(S) = 0.3855x^2 - 3.3352x + 5.4069
\qquad \text{where} \qquad
x = \frac{S - 4000}{1000}
$
    
| Distance [cm] | Sensor Value | Computed Distance [cm] |
|:---------------------------:|:---------------------:|:-------------------------------:|
|              <1             |          5400         |             1.49320             |
|              1              |          5330         |             1.65299             |
|              2              |          5267         |             1.80001             |
|              3              |          5014         |             2.42138             |
|              4              |          4503         |             3.82682             |
|              5              |          4130         |             4.97984             |
|              6              |          3763         |             6.21899             |
|              7              |          3471         |             7.27909             |
|              8              |          3241         |             8.16039             |
|              9              |          3011         |             9.08247             |
|              10             |          2844         |             9.77754             |
|              11             |          2640         |             10.6557             |
|              12             |          2330         |             12.0518             |
|              13             |          2109         |             13.0922             |
|              14             |            0          |                -                |

Based on those values, we choose `FOLLOW_THRESHOLD = 3000` (~9cm), and `PARKING_THRESHOLD = 4600` (~3.54cm)

The ground sensor threshold is calibrated separately by measuring the sensor values on the white surface and black boundary. Doing that we defined `GROUND_THRESHOLD = 500`, knowing that full black is 200 and full white around 1000.

### 6.4. Avoidance strategy

When an obstacle is detected, the module compares the left and right proximity measurements and turns towards the side with the lower obstacle intensity.

In [9]:
def choose_direction(prox, threshold):
    left_strength = sum(
        max(0, value - threshold) for value in prox[0:2]
    )
    right_strength = sum(
        max(0, value - threshold) for value in prox[3:5]
    )

    if left_strength > right_strength:
        return "right"
    elif right_strength > left_strength:
        return "left"
    else:
        return "right" #right by default

To illustrate this behaviour, we consider three simulated situations : an obstacle on the left, an obstacle on the right and one directly in front of the robot :

In [8]:
threshold = 3000

# Case 1: Obstacle on the left
prox_left = [4500, 3800, 1000, 1200, 1000, 0, 0]

# Case 2: Obstacle on the right
prox_right = [1000, 1200, 1000, 3800, 4500, 0, 0]

# Case 3: Obstacle in front
prox_front = [1000, 1000, 4500, 1000, 1000, 0, 0]

print("Case 1 - direction chosen :", choose_direction(prox_left, threshold))
print("Case 2 - direction chosen :", choose_direction(prox_right, threshold))
print("Case 3 - direction chosen :", choose_direction(prox_front, threshold))

Case 1 - direction chosen : right
Case 2 - direction chosen : left
Case 3 - direction chosen : right


An internal state and a counter ensure that avoidance continues until several consecutive measurements indicate that the obstacle has been cleared. This prevents premature switching back to global navigation.

In [11]:
CLEAR_LIMIT = 3
if not _avoiding and obstacle:
    _avoiding = True
    _clear_count = 0
    _direction = choose_direction(prox, threshold)

if _avoiding:
    if obstacle:
        _clear_count = 0
    else:
        _clear_count += 1

    if _clear_count >= CLEAR_LIMIT:
        _avoiding = False
        _clear_count = 0
        return None

    return motor_command(_direction)

return None

NameError: name '_avoiding' is not defined

The module can return three types of outputs:

- `None`: global navigation retains control.
- `(left, right)`: local navigation overrides motor commands.
- `(0, 0)`: the black boundary is detected and the robot stops.

### 6.5. Limitations and improvements

The current strategy is deliberately simple and reactive. Potential improvements include individual sensor calibration, smoother avoidance trajectories and more reliable recovery towards the global path.

One important limitation is that proximity sensors cannot distinguish parked vehicles from unexpected obstacles. Using different thresholds during following and parking reduces unnecessary avoidance, but does not completely solve this problem.

### 6.6. Conclusion

The local navigation module provides a reactive layer that complements global path planning. Its simple architecture and configurable thresholds allow it to handle unexpected obstacles while remaining easy to integrate and calibrate.

## 7. Filtering

*To write (Joshua): theory, choices, measurements, parameters. Show the pose before and after filtering.*

## 8. Full run

*To write: run the whole project (or point to `main.py`). Show the path chosen and where the system believes the robot is, before and after filtering.*

## 9. Conclusion

*To write.*

## Sources

*To write: everything we used.*